# Monster Hunting: a step-by-step anomaly notebook

This notebook is intentionally simple and explicit.

We will:

1. load the **actual training tables**;
2. inspect what each table contains;
3. create one clean activity table from video and LMS events;
4. run each monster rule separately;
5. explain what each result means;
6. avoid claiming findings when the required table is unavailable.

The available row-level training files are:

- `la_mmh_training_video_activity_by_view_fact.csv`
- `la_mmh_training_lms_item_activity_by_session_fact.csv`
- `la_mmh_training_person_dim.csv`
- `la_mmh_training_person_module_instance_fact.csv`

The other files in the folder are data dictionaries, not row-level data.

## Important definitions

An anomaly result is a **candidate**, not automatically a confirmed problem.

- **Phantom:** repeated activity records.
- **Ghost:** active in at least 3 of the first 4 Autumn weeks, then below 10% of the earlier weekly average. Requires the real calendar table.
- **Zombie:** activity continues for more than 3 hours, with no gap longer than 15 minutes.
- **Vampire:** more active days during 00:00–05:00 than during 05:00–24:00, with more than 21 night-active days.
- **Undead Student:** activity after the student's deregistration date.
- **Doppelgänger:** students with identical activity profiles. The notebook uses the simple video-only rule matching the challenge-style code.
- **Shape-shifter:** programme/module change. Requires the real module dimension table.

In [ ]:
from pathlib import Path
import itertools
import math
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Change only this path if your folder is elsewhere.
DATA_DIR = Path(r"C:\Users\cvill\Downloads\Training (1)\Training")

print("Folder exists:", DATA_DIR.exists())
print("Folder:", DATA_DIR)


## 1. Load the real training tables

We use explicit filenames here. This prevents the notebook from accidentally loading the similarly named data-dictionary files.

In [ ]:
FILE_MAP = {
    "video_activity": "la_mmh_training_video_activity_by_view_fact.csv",
    "lms_activity": "la_mmh_training_lms_item_activity_by_session_fact.csv",
    "person": "la_mmh_training_person_dim.csv",
    "person_module": "la_mmh_training_person_module_instance_fact.csv",
}

tables = {}

for table_name, filename in FILE_MAP.items():
    path = DATA_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)
    df.columns = [str(c).replace("\ufeff", "").strip() for c in df.columns]
    tables[table_name] = df

    print(f"{table_name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print("  columns:", list(df.columns))


### What to check

The two activity tables should contain `P_ID`, `DATE`, and `TIMESTAMP`.

The person table should contain `P_ID` and `deregistration_date`.

If the columns shown here are `field_name`, `data_type`, and `example_data`, you loaded a data dictionary instead of the training file.

In [ ]:
for name, df in tables.items():
    print(f"\n{name}")
    display(df.head(3))


## 2. Standardise the activity data

In [ ]:
def clean_column_name(column):
    column = str(column).replace("\ufeff", "").replace("\xa0", " ")
    column = column.strip().upper()
    column = re.sub(r"[^A-Z0-9]+", "_", column)
    return column.strip("_")


def prepare_columns(df):
    out = df.copy()
    out.columns = [clean_column_name(c) for c in out.columns]
    return out


def standardise_activity(df, source):
    out = prepare_columns(df)
    out["_SOURCE"] = source

    if "P_ID" not in out.columns:
        raise KeyError(f"{source}: P_ID is missing")
    if "TIMESTAMP" not in out.columns:
        raise KeyError(f"{source}: TIMESTAMP is missing")
    if "DATE" not in out.columns:
        raise KeyError(f"{source}: DATE is missing")

    out["_P_ID"] = out["P_ID"].astype("string")
    out["_TIMESTAMP"] = pd.to_datetime(
        out["TIMESTAMP"], errors="coerce", utc=True
    )
    out["_DATE"] = pd.to_datetime(
        out["DATE"], errors="coerce", utc=True
    ).dt.normalize()

    return out


video = standardise_activity(tables["video_activity"], "video")
lms = standardise_activity(tables["lms_activity"], "lms")

activity = pd.concat([video, lms], ignore_index=True, sort=False)

print("Video rows:", f"{len(video):,}")
print("LMS rows:", f"{len(lms):,}")
print("Combined activity rows:", f"{len(activity):,}")

display(activity[["_SOURCE", "_P_ID", "_TIMESTAMP", "_DATE"]].head())


This creates one common activity table.

- `_SOURCE` says whether a row came from video or LMS.
- `_P_ID` is the common student ID.
- `_TIMESTAMP` is the event time.
- `_DATE` is the event date.

No anomaly has been detected yet.

## 3. Basic data-quality checks

In [ ]:
quality = pd.DataFrame({
    "rows": [len(video), len(lms), len(activity)],
    "missing_P_ID": [
        video["_P_ID"].isna().sum(),
        lms["_P_ID"].isna().sum(),
        activity["_P_ID"].isna().sum(),
    ],
    "missing_timestamp": [
        video["_TIMESTAMP"].isna().sum(),
        lms["_TIMESTAMP"].isna().sum(),
        activity["_TIMESTAMP"].isna().sum(),
    ],
    "missing_date": [
        video["_DATE"].isna().sum(),
        lms["_DATE"].isna().sum(),
        activity["_DATE"].isna().sum(),
    ],
}, index=["video", "lms", "combined"])

display(quality)
print("Unique students in activity:", activity["_P_ID"].nunique())
print("Activity date range:", activity["_DATE"].min(), "to", activity["_DATE"].max())


## 4. Undead Student

Rule: keep activity whose date is **strictly after** the student's deregistration date.

Activity on the deregistration date itself is not flagged.

In [ ]:
people = prepare_columns(tables["person"])
people["P_ID"] = people["P_ID"].astype("string")
people["DEREGISTRATION_DATE"] = pd.to_datetime(
    people["DEREGISTRATION_DATE"], errors="coerce", utc=True
).dt.normalize()

# Check whether person_dim has one row per student.
print("Person rows:", len(people))
print("Unique P_IDs:", people["P_ID"].nunique())
print("Duplicate P_ID rows:", people["P_ID"].duplicated().sum())

undead = activity.copy()
undead["_ACTIVITY_DATE"] = undead["_TIMESTAMP"].dt.normalize()

undead = undead.merge(
    people[["P_ID", "DEREGISTRATION_DATE"]],
    left_on="_P_ID",
    right_on="P_ID",
    how="left"
)

undead = undead[
    undead["DEREGISTRATION_DATE"].notna()
    & undead["_ACTIVITY_DATE"].notna()
    & (undead["_ACTIVITY_DATE"] > undead["DEREGISTRATION_DATE"])
].copy()

undead_summary = (
    undead.groupby(["_P_ID", "DEREGISTRATION_DATE"], as_index=False)
    .agg(
        SOURCES=("_SOURCE", lambda s: ", ".join(sorted(s.unique()))),
        POST_DEREG_ACTIVITY_ROWS=("_TIMESTAMP", "size"),
        FIRST_POST_DEREG_ACTIVITY=("_TIMESTAMP", "min"),
        LAST_POST_DEREG_ACTIVITY=("_TIMESTAMP", "max"),
    )
    .sort_values("POST_DEREG_ACTIVITY_ROWS", ascending=False)
)

print(f"Undead candidates: {len(undead_summary):,}")
display(undead_summary.head(20))


Interpretation: each row in `undead_summary` is a student with at least one activity date after deregistration.

Before treating this as a real anomaly, check duplicate `P_ID`s and possible date/time-zone issues.

## 5. Phantom

We run two clearly separated checks:

1. **Exact duplicate rows:** compare every original field, including technical IDs.
2. **Repeated activity patterns:** ignore technical IDs and compare behavioural fields.

The first check respects the fact that IDs may be important. The second is exploratory and must not be called an exact duplicate check.

In [ ]:
def exact_duplicate_groups(df, source):
    # Keep every original column, including PVT_ID or PIT_ID.
    cols = [c for c in df.columns if not c.startswith("_")]

    duplicated = df[df.duplicated(subset=cols, keep=False)].copy()

    if duplicated.empty:
        return pd.DataFrame()

    result = (
        duplicated.groupby(cols, dropna=False, sort=False)
        .size()
        .reset_index(name="DUPLICATE_COUNT")
        .sort_values("DUPLICATE_COUNT", ascending=False)
    )
    result.insert(0, "SOURCE", source)
    return result


phantom_exact = pd.concat([
    exact_duplicate_groups(video, "video"),
    exact_duplicate_groups(lms, "lms"),
], ignore_index=True, sort=False)

print(f"Exact duplicate groups: {len(phantom_exact):,}")
display(phantom_exact.head(20))


In [ ]:
def repeated_activity_patterns(df, source):
    ignored = {"PVT_ID", "PIT_ID", "PMI_ID", "LOGIN_ID", "LOG_IN_ID"}
    cols = [
        c for c in df.columns
        if c not in ignored and not c.startswith("_")
    ]

    duplicated = df[df.duplicated(subset=cols, keep=False)].copy()

    if duplicated.empty:
        return pd.DataFrame()

    result = (
        duplicated.groupby(cols, dropna=False, sort=False)
        .size()
        .reset_index(name="REPEATED_PATTERN_COUNT")
        .sort_values("REPEATED_PATTERN_COUNT", ascending=False)
    )
    result.insert(0, "SOURCE", source)
    return result


phantom_patterns = pd.concat([
    repeated_activity_patterns(video, "video"),
    repeated_activity_patterns(lms, "lms"),
], ignore_index=True, sort=False)

print(f"Repeated behavioural patterns: {len(phantom_patterns):,}")
display(phantom_patterns.head(20))


The two counts answer different questions.

- `phantom_exact`: the complete original row repeats.
- `phantom_patterns`: the activity description repeats after technical IDs are ignored.

Neither result alone proves that the repeated record is erroneous.

## 6. Doppelgänger

This version follows the simple challenge-style rule:

- use the video table;
- define each event as `VIDEO_ID + DATE + TIMESTAMP`;
- sort each student's complete event list;
- identify identical student profiles.

The output counts groups and students separately.

In [ ]:
video_d = video.dropna(
    subset=["_P_ID", "VIDEO_ID", "_DATE", "_TIMESTAMP"]
).copy()

video_d["EVENT_SIGNATURE"] = (
    video_d["VIDEO_ID"].astype("string")
    + "|"
    + video_d["_DATE"].dt.strftime("%Y-%m-%d")
    + "|"
    + video_d["_TIMESTAMP"].astype("string")
)

student_profiles = (
    video_d.groupby("_P_ID")["EVENT_SIGNATURE"]
    .apply(lambda values: tuple(sorted(values.tolist())))
    .reset_index(name="PROFILE")
)

shared_profiles = student_profiles[
    student_profiles.duplicated(subset=["PROFILE"], keep=False)
].copy()

doppel_groups = (
    shared_profiles.groupby("PROFILE")["_P_ID"]
    .agg(list)
    .reset_index(name="P_IDS")
)

doppel_groups["NUMBER_OF_STUDENTS"] = doppel_groups["P_IDS"].str.len()
doppel_groups["NUMBER_OF_PAIRS"] = doppel_groups["NUMBER_OF_STUDENTS"].apply(
    lambda n: math.comb(n, 2)
)

print("Shared profile groups:", len(doppel_groups))
print("Students involved:", shared_profiles["_P_ID"].nunique())
print("All possible student pairs:", doppel_groups["NUMBER_OF_PAIRS"].sum())

display(doppel_groups.head(20))


In [ ]:
# Submission-style output: one P_ID per row, each student only once.
doppel_submission = (
    shared_profiles[["_P_ID"]]
    .drop_duplicates()
    .rename(columns={"_P_ID": "P_ID"})
    .sort_values("P_ID")
)

display(doppel_submission.head(20))

# Optional export
doppel_submission.to_csv("doppelganger_pids.csv", index=False)
print("Saved: doppelganger_pids.csv")


## 7. Zombie

Rule: for a student, activity timestamps form a continuous run when consecutive events are no more than 15 minutes apart. A run is a Zombie candidate when it lasts more than 3 hours.

This uses both video and LMS activity.

In [ ]:
z = activity.dropna(subset=["_P_ID", "_TIMESTAMP"]).copy()
z = z[["_P_ID", "_TIMESTAMP", "_SOURCE"]].sort_values(["_P_ID", "_TIMESTAMP"])

z["GAP_MINUTES"] = (
    z.groupby("_P_ID")["_TIMESTAMP"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

# A new run starts after a gap greater than 15 minutes.
z["RUN_ID"] = (
    z["GAP_MINUTES"].isna()
    | (z["GAP_MINUTES"] > 15)
).groupby(z["_P_ID"]).cumsum()

zombie_runs = (
    z.groupby(["_P_ID", "RUN_ID"], as_index=False)
    .agg(
        START=("_TIMESTAMP", "min"),
        END=("_TIMESTAMP", "max"),
        ACTIVITY_ROWS=("_TIMESTAMP", "size"),
        SOURCES=("_SOURCE", lambda s: ", ".join(sorted(s.unique()))),
    )
)

zombie_runs["DURATION_HOURS"] = (
    zombie_runs["END"] - zombie_runs["START"]
).dt.total_seconds() / 3600

zombie_candidates = zombie_runs[
    zombie_runs["DURATION_HOURS"] > 3
].sort_values("DURATION_HOURS", ascending=False)

print(f"Zombie runs: {len(zombie_candidates):,}")
print(f"Zombie students: {zombie_candidates['_P_ID'].nunique():,}")
display(zombie_candidates.head(20))


## 8. Vampire

Rule:

- night is 00:00 inclusive to 05:00 exclusive;
- day is 05:00 inclusive to midnight;
- retain students with more than 21 night-active days;
- retain students with more night-active days than day-active days.

In [ ]:
v = activity.dropna(subset=["_P_ID", "_TIMESTAMP"]).copy()
v["ACTIVITY_DATE"] = v["_TIMESTAMP"].dt.normalize()
v["HOUR"] = v["_TIMESTAMP"].dt.hour
v["PERIOD"] = np.where(v["HOUR"] < 5, "night", "day")

days = (
    v.groupby(["_P_ID", "PERIOD"])["ACTIVITY_DATE"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["night", "day"]:
    if col not in days.columns:
        days[col] = 0

vampire_candidates = days[
    (days["night"] > 21)
    & (days["night"] > days["day"])
].copy()

vampire_candidates = vampire_candidates.sort_values(
    ["night", "day"], ascending=[False, True]
)

print(f"Vampire students: {len(vampire_candidates):,}")
display(vampire_candidates.head(20))


## 9. Ghost

Ghost requires the actual calendar dimension with at least:

- `DATE`;
- `WEEK_NUMBER`;
- `TERM`.

In this dataset folder, `la_mmh_calendar_dim.csv` is a data dictionary, not row-level calendar data. Therefore we do not invent Autumn weeks or run this rule.

In [ ]:
ghost_result = {
    "status": "not_evaluated",
    "reason": (
        "The folder contains the calendar data dictionary but not the "
        "row-level calendar dimension. Autumn weeks cannot be identified reliably."
    ),
    "candidates": pd.DataFrame(),
}

print("Ghost:", ghost_result["status"])
print(ghost_result["reason"])


## 10. Shape-shifter

Shape-shifter requires the actual module dimension, especially a programme/department field such as `DEPARTMENT_ID`.

The available `person_module_instance_fact` table tells us which module instances a person registered for, but without the real module dimension we cannot determine whether the programme changed. We therefore report this as unavailable rather than guessing.

In [ ]:
shape_shifter_result = {
    "status": "not_evaluated",
    "reason": (
        "The actual module_instance_dim table is unavailable. "
        "Programme changes cannot be inferred reliably from the available tables."
    ),
    "candidates": pd.DataFrame(),
}

print("Shape-shifter:", shape_shifter_result["status"])
print(shape_shifter_result["reason"])


## 11. Final result summary

This table records what was actually evaluated, using the definitions in this notebook.

In [ ]:
summary = pd.DataFrame([
    {
        "MONSTER": "Phantom exact",
        "STATUS": "evaluated",
        "COUNT": len(phantom_exact),
        "UNIT": "duplicate groups",
        "DEFINITION": "complete original activity rows repeat",
    },
    {
        "MONSTER": "Phantom behavioural",
        "STATUS": "evaluated",
        "COUNT": len(phantom_patterns),
        "UNIT": "repeated groups",
        "DEFINITION": "activity repeats after technical IDs are ignored",
    },
    {
        "MONSTER": "Doppelganger",
        "STATUS": "evaluated",
        "COUNT": len(doppel_groups),
        "UNIT": "shared video profiles",
        "DEFINITION": "identical VIDEO_ID + DATE + TIMESTAMP profiles",
    },
    {
        "MONSTER": "Zombie",
        "STATUS": "evaluated",
        "COUNT": len(zombie_candidates),
        "UNIT": "continuous runs",
        "DEFINITION": "more than 3 hours with no gap over 15 minutes",
    },
    {
        "MONSTER": "Vampire",
        "STATUS": "evaluated",
        "COUNT": len(vampire_candidates),
        "UNIT": "students",
        "DEFINITION": "more than 21 night days and more night than day days",
    },
    {
        "MONSTER": "Undead Student",
        "STATUS": "evaluated",
        "COUNT": len(undead_summary),
        "UNIT": "students",
        "DEFINITION": "activity strictly after deregistration",
    },
    {
        "MONSTER": "Ghost",
        "STATUS": "not evaluated",
        "COUNT": np.nan,
        "UNIT": "students",
        "DEFINITION": "requires row-level calendar data",
    },
    {
        "MONSTER": "Shape-shifter",
        "STATUS": "not evaluated",
        "COUNT": np.nan,
        "UNIT": "students",
        "DEFINITION": "requires row-level module dimension data",
    },
])

display(summary)


## What this notebook does not do

It does not prove that a student cheated or that a row is wrong.

It produces transparent candidate lists that can later be used to:

- inspect the source records;
- validate the anomaly definitions;
- create labelled training data;
- build a predictive model for unseen data.

Every candidate should be validated against the underlying event records before being treated as a confirmed anomaly.